# Depth revision code


In [1]:
import flexiznam as flz
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import json
import os
from tqdm import tqdm
import pickle
from scipy.stats import gaussian_kde
import cottage_analysis.analysis.is_learned as learn


AttributeError: Invalid project: "colasa_3d-vision_revisions"

### Is depth selectivity innate?

In [ ]:
PROJECT='colasa_3d-vision_revisions'
SESSION='PZAG16.3c_S20250219'

In [ ]:
flz_session = flz.get_flexilims_session(PROJECT)

In [ ]:
suite2p_datasets = flz.get_datasets(
        origin_name=SESSION,
        dataset_type="suite2p_rois",
        project_id=PROJECT,
        flexilims_session=flz_session,
        return_dataseries=False,
    )

In [ ]:
dataset = suite2p_datasets[0]

In [ ]:
path = Path('/nemo/lab/znamenskiyp/home/shared/projects/colasa_3d-vision_revisions/PZAG16.3c/S20250219')

In [ ]:
datapath = path / 'neurons_df.pickle'

In [ ]:
neurons_df = pd.read_pickle(datapath)

In [ ]:
neurons_df.columns

In [ ]:
plt.hist(neurons_df['depth_neuron_anova_p'], bins = 100)
plt.axvline(0.05, color = 'red')

In [ ]:
plt.hist(neurons_df['preferred_depth_closedloop'][neurons_df['is_depth_neuron']==True])

In [ ]:
plt.hist(neurons_df['preferred_depth_closedloop'][neurons_df['depth_neuron_anova_p']<0.00001])

In [ ]:
sig_neurons_df = neurons_df[neurons_df['is_depth_neuron']].copy()
sig_neurons_df['log_pref_depth'] = np.log10(neurons_df['preferred_depth_closedloop']*100)
plt.hist(sig_neurons_df['log_pref_depth'], bins = 100)
plt.ylabel('Frequency')
plt.xlabel('log preferred depth (cm)')
plt.title('PZAG16.3c, session 0, significantly depth tuned neurons')

In [ ]:
plt.scatter(neurons_df['preferred_depth_closedloop'], neurons_df['depth_neuron_anova_p'], alpha = 0.5)

In [ ]:

# Count occurrences of True and False in the 'is_depth_neuron' column
value_counts = neurons_df['is_depth_neuron'].value_counts()

# Create a bar plot
plt.figure(figsize=(6, 4))
plt.bar(value_counts.index.astype(str), value_counts.values)

In [ ]:
plt.hist(sig_neurons_df['depth_tuning_test_spearmanr_rval_closedloop'])

In [ ]:
plt.hist(neurons_df['depth_tuning_test_spearmanr_rval_closedloop'])

## Trying the baseline dist

In [ ]:
baseline_datapath = Path('/nemo/lab/znamenskiyp/home/shared/projects/hey2_3d-vision_foodres_20220101/PZAH8.2i/S20230209')

In [ ]:
bas_datapath = baseline_datapath / 'neurons_df.pickle'

In [ ]:
baseline_df = pd.read_pickle(bas_datapath)

In [ ]:
baseline_df.columns

In [ ]:
plt.hist(baseline_df['preferred_depth_closedloop'])

In [ ]:
plt.hist(baseline_df['preferred_depth_closedloop'][baseline_df['is_depth_neuron']==True])

In [ ]:
bas_sig_neurons_df = baseline_df[baseline_df['is_depth_neuron']].copy()
bas_sig_neurons_df['log_pref_depth'] = np.log10(bas_sig_neurons_df['preferred_depth_closedloop']*100)
plt.hist(bas_sig_neurons_df['log_pref_depth'], bins = 100)
plt.ylabel('Frequency')
plt.xlabel('log preferred depth (cm)')
plt.title('PZAH8.2i, session N, significantly depth tuned neurons')

# Finding back the ROIs

In [ ]:
datapath = dataset.path_full / 'plane0'

In [ ]:
iscell = np.load(datapath / 'iscell.npy')

In [ ]:
iscell.shape

# Selectivity across days (no ROIs)

- We want to plot the histograms of depth selectivities over days, to check if there are any changes.
- That is, however we plot it, a dataframe that has sessions in columns, nday, mouse, and a set of bins that have a set proportion of cells.
- Access all the dataframes of mice, then bin the depths and keep those numbers



In [ ]:
PROJECT='colasa_3d-vision_revisions'
flz_session = flz.get_flexilims_session(PROJECT)


In [ ]:
micelist = ['PZAG16.3b', 'PZAG16.3c', 'PZAH17.1e']

### Thinking about how to build the code

In [ ]:
sessions = flz.get_children(
    parent_name = micelist[0], 
    children_datatype = 'session',
    project_id = PROJECT, 
    flexilims_session= flz_session
)
        

In [ ]:
processed_root = flz.get_data_root('processed', 
                                   project=PROJECT, 
                                   flexilims_session=flz_session
                                  )


In [ ]:
sesspath = processed_root / sessions.path[0]

In [ ]:
SphereTube_recordings = flz.get_children(
    parent_name = sessions.name[0], 
    children_datatype = 'recording', 
    project_id=PROJECT, 
    flexilims_session=flz_session
)


In [ ]:
SphereTube_recordings = SphereTube_recordings[SphereTube_recordings['protocol']=='SpheresPermTubeReward']

In [ ]:
recordings = []
for i in sessions.name:
    suite2p_datasets = flz.get_datasets(
        origin_name=i,
        dataset_type="suite2p_rois",
        project_id=PROJECT,
        flexilims_session=flz_session,
        return_dataseries=False,
    )
    print(suite2p_datasets)
    if suite2p_datasets != []:
        recordings.append(suite2p_datasets)
    

In [ ]:
recordings[0][0].path.parent

In [ ]:
def find_processed_sessions(mouse, protocol = 'SpheresPermTubeReward'):
    #Check all sessions
    sessions = flz.get_children(
        parent_name = mouse, 
        children_datatype = 'session',
        project_id = PROJECT, 
        flexilims_session= flz_session
    )
    #print(sessions.name)
    #List the sessions that are SphereTube
    for i in sessions.name:
        SphereTube_recordings = flz.get_children(
            parent_name = i, 
            children_datatype = 'recording', 
            project_id=PROJECT, 
            flexilims_session=flz_session
        )
        SphereTube_recordings = SphereTube_recordings[SphereTube_recordings['protocol']=='SpheresPermTubeReward']
        if len(SphereTube_recordings)==0:
            sessions = sessions[sessions['name']!= i]

    #Keep the sessions that are processed
    for i, session in sessions.iterrows():
        #print(session)
        neurons_path = processed_root / session.path / 'neurons_df.pickle'
        if not os.path.isfile(neurons_path):
            name_to_drop = session.name
            sessions = sessions[sessions['name']!= name_to_drop]
            
    print(sessions.name)
    return sessions

In [ ]:
mouse_i = 0
for mouse in micelist:
    if mouse_i==0:
        processed = find_processed_sessions(mouse)
    else:
        processed = pd.concat([find_processed_sessions(mouse), processed], ignore_index=True)
    mouse_i += 1
    
processed

In [ ]:
def save_significant_neurons(neurons_df):
    '''
    
    '''
    # Filter significantly depth-tuned neurons
    sig_neurons_df = neurons_df[neurons_df['is_depth_neuron']].copy()

    # Compute log preferred depth (in cm)
    sig_neurons_df['log_pref_depth'] = np.log10(sig_neurons_df['preferred_depth_closedloop'] * 100)

    return sig_neurons_df

def save_histogram(sig_neurons_df, session, path, print_figure = True):
    """
    Saves a histogram of the log preferred depth of significantly depth-tuned neurons 
    and returns an alternative histogram with 10 bins as a NumPy array.
    
    Args:
        neurons_df (pd.DataFrame): DataFrame containing neuron data.
        mouse (str): Mouse identifier.
        session (int): Session number.
        path (str, optional): Directory to save the figure. Defaults to "figures/".
    
    Returns:
        np.array: Histogram data with 10 bins.
    """

    # Ensure path exists
    save_path = Path(path)
    save_path.mkdir(parents=True, exist_ok=True)

    if print_figure:
        # Save the histogram with 100 bins
        plt.figure(figsize=(8, 6))
        plt.hist(sig_neurons_df['log_pref_depth'], bins=100, color="blue", alpha=0.7)
        plt.ylabel('Frequency')
        plt.xlabel('Log Preferred Depth (cm)')
        plt.title(f'Session {session}, Significantly Depth-Tuned Neurons')
        
        # Save figure
        filename = save_path / f"session_{session}_hist.png"
        plt.savefig(filename, dpi=300)
        plt.close()

    # Generate alternative histogram with 10 bins
    hist_counts, bin_edges = np.histogram(sig_neurons_df['log_pref_depth'], bins=10)

    return hist_counts, bin_edges

class NumpyFixUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == 'numpy._core.numeric':
            module = 'numpy.core.numeric'
        return super().find_class(module, name)


In [ ]:
hist_counts = []
bin_edges = []
mean_log_depths = []
sems = []

for i, session in tqdm(processed.iterrows()):
    neurons_path = processed_root / session.path / 'neurons_df.pickle'
    save_path = processed_root / session.path / 'figures/'

    with open(neurons_path, 'rb') as f:
        neurons_df = NumpyFixUnpickler(f).load()

    sig_neurons_df = save_significant_neurons(neurons_df)

    # Compute mean and sem of log_pref_depth for significant neurons
    mean_log_depth = np.mean(sig_neurons_df['log_pref_depth'])
    sem = np.std(sig_neurons_df['log_pref_depth'], ddof=1) / np.sqrt(len(sig_neurons_df))

    # Save values
    mean_log_depths.append(mean_log_depth)
    sems.append(sem)

    # Get histogram
    hist_count, bin_edge = save_histogram(
        sig_neurons_df, session.name, save_path, print_figure=False
    )
    hist_counts.append(hist_count)
    bin_edges.append(bin_edge)

# Assign all data to the dataframe
processed['hist_counts'] = hist_counts
processed['bin_edges'] = bin_edges
processed['mean_log_depth'] = mean_log_depths
processed['sem_log_depth'] = sems




In [ ]:
name = processed.name[0]
namelist = name.split('_')
mouse = namelist[0]
date = namelist[1]

In [ ]:
mice = []
dates = []
for i, session in processed.iterrows():
    name = session['name']
    namelist = name.split('_')
    mice.append(namelist[0])
    dates.append(namelist[1])

processed['mouse'] = mice
processed['date'] =  dates

processed_mice = list(set(processed['mouse']))


In [ ]:

# Ensure 'date' is treated as a string or int for sorting (yyyymmdd format is naturally sortable)
processed = processed.sort_values(by=['mouse', 'date']).reset_index(drop=True)

# Create the exposure_day column by grouping by mouse and ranking the date
processed['exposure_day'] = (
    processed
    .groupby('mouse')['date']
    .rank(method='dense')  # or method='first' if dates could repeat
    .astype(int) - 1        # Start from 0
)

In [ ]:
def build_sessions_df(micelist):
    
    #generate the barebones processed dataframe
    mouse_i = 0
    for mouse in micelist:
        if mouse_i==0:
            processed = find_processed_sessions(mouse)
        else:
            processed = pd.concat([find_processed_sessions(mouse), processed], ignore_index=True)
        mouse_i += 1
    
    #add variable values
    hist_counts = []
    bin_edges = []
    mean_log_depths = []
    sems = []

    for i, session in tqdm(processed.iterrows()):
        neurons_path = processed_root / session.path / 'neurons_df.pickle'
        save_path = processed_root / session.path / 'figures/'

        with open(neurons_path, 'rb') as f:
            neurons_df = NumpyFixUnpickler(f).load()

        sig_neurons_df = save_significant_neurons(neurons_df)

        # Compute mean and sem of log_pref_depth for significant neurons
        mean_log_depth = np.mean(sig_neurons_df['log_pref_depth'])
        sem = np.std(sig_neurons_df['log_pref_depth'], ddof=1) / np.sqrt(len(sig_neurons_df))

        # Save values
        mean_log_depths.append(mean_log_depth)
        sems.append(sem)

        # Get histogram
        hist_count, bin_edge = save_histogram(
            sig_neurons_df, session.name, save_path, print_figure=False
        )
        hist_counts.append(hist_count)
        bin_edges.append(bin_edge)

    # Assign all data to the dataframe
    processed['hist_counts'] = hist_counts
    processed['bin_edges'] = bin_edges
    processed['mean_log_depth'] = mean_log_depths
    processed['sem_log_depth'] = sems

    #Add mice and date names
    mice = []
    dates = []
    for i, session in processed.iterrows():
        name = session['name']
        namelist = name.split('_')
        mice.append(namelist[0])
        dates.append(namelist[1])

    processed['mouse'] = mice
    processed['date'] =  dates


    #Add exposure dates

    # Ensure 'date' is treated as a string or int for sorting (yyyymmdd format is naturally sortable)
    processed = processed.sort_values(by=['mouse', 'date']).reset_index(drop=True)

    # Create the exposure_day column by grouping by mouse and ranking the date
    processed['exposure_day'] = (
        processed
        .groupby('mouse')['date']
        .rank(method='dense')  # or method='first' if dates could repeat
        .astype(int) - 1        # Start from 0
    )

    return processed

In [ ]:

fig, ax = plt.subplots()

for mouse in tqdm(processed_mice):
    mouse_data = processed[processed['mouse'] == mouse]
    ax.errorbar(
        mouse_data['exposure_day'],
        mouse_data['mean_log_depth'],
        yerr=mouse_data['sem_log_depth'],
        label=mouse,
        capsize=3,           # small caps on error bars
        marker='o',          # dots on each point
        linestyle='-',       # connect the points
        linewidth=1
    )

ax.set_xlabel('Exposure Day')
ax.set_ylabel('Mean Log Depth')
ax.set_title('Mean Log Depth over Exposure Days by Mouse')
ax.legend()
plt.tight_layout()
plt.show()



In [ ]:
neurons_df.columns

### Plotting

In [ ]:
processed = learn.build_sessions_df(micelist)

In [ ]:
learn.plot_spearman_r_kde_by_day(processed)


Plot proportion of depth selective neurons per day

For this, 

In [ ]:
def plot_selective_proportion_over_days(processed, mice=None):
    """
    Plot proportion of depth-tuned (selective) neurons over exposure days for each mouse.

    Parameters:
    - processed: DataFrame with columns ['mouse', 'exposure_day', 'proportion_depthtuned']
    - mice: optional list of mouse IDs to plot (default: all unique mice in `processed`)
    
    Returns:
    - fig: the matplotlib figure object
    """
    if mice is None:
        mice = processed['mouse'].unique()

    fig, ax = plt.subplots()

    for mouse in tqdm(mice, desc="Plotting mice"):
        mouse_data = processed[processed['mouse'] == mouse]

        ax.plot(
            mouse_data['exposure_day'],
            mouse_data['proportion_depthtuned'],
            label=mouse,
            marker='o',
            linestyle='-',
            linewidth=1
        )

    ax.set_xlabel('Exposure Day')
    ax.set_ylabel('Proportion of Depth-Tuned Neurons')
    ax.set_title('Proportion of Selective Neurons over Exposure Days by Mouse')
    ax.legend()
    plt.tight_layout()
    plt.show()

    return fig

In [ ]:
learn.plot_selective_proportion_over_days(processed)


Plot the distributions of depth-tuned neurons

(bear in mind that gaussian KDEs are not ideal to plot non-unimodal distributions)

In [ ]:
def plot_log_pref_depth_kde_by_day(processed):
    """One subplot per exposure day showing KDEs of log_pref_depth for all mice."""
    
    exposure_days = sorted(processed['exposure_day'].unique())
    mice = sorted(processed['mouse'].unique())
    colors = plt.cm.tab10(np.linspace(0, 1, len(mice)))  # consistent colors per mouse
    mouse_color_map = dict(zip(mice, colors))

    fig, axes = plt.subplots(len(exposure_days), 1, figsize=(10, 2.5 * len(exposure_days)), sharex=True)

    if len(exposure_days) == 1:
        axes = [axes]  # Make iterable if only one subplot

    for ax, day in zip(axes, exposure_days):
        day_data = processed[processed['exposure_day'] == day]

        for _, row in day_data.iterrows():
            mouse = row['mouse']
            color = mouse_color_map[mouse]
            dist = row['log_pref_depth']

            if len(dist) < 2:
                continue  # can't KDE on 1 point

            kde = gaussian_kde(dist)
            x_range = np.linspace(min(dist), max(dist), 200)
            kde_vals = kde(x_range)

            # Plot KDE
            ax.plot(x_range, kde_vals, label=mouse, color=color, alpha=0.7)

            # Plot vertical line at median
            ax.axvline(np.median(dist), color=color, linestyle='--', alpha=0.7)

        ax.set_ylabel(f"Day {day}")
        ax.grid(True)

    axes[-1].set_xlabel("log(pref depth)")
    axes[0].set_title("KDE of log(pref depth) by Mouse for Each Exposure Day")

    # Legend: only once
    handles = [plt.Line2D([0], [0], color=mouse_color_map[m], label=m) for m in mice]
    axes[0].legend(handles=handles, title="Mouse", bbox_to_anchor=(1.05, 1), loc="upper left")

    plt.tight_layout()
    plt.show()

    return fig

In [ ]:
plot_log_pref_depth_kde_by_day(processed)

## Quantofying significance of differences

First, we look at the distribution of spearman's r's, and compare for each mouse if it changes over days 

In [ ]:
mice = None
from scipy.stats import mannwhitneyu

def test_changes_over_days(processed, property = 'spearman_r_dist', mice = None):
        if mice is None:
                mice = processed['mouse'].unique()

        p_values = []
        u_values = []

        for mouse in mice:
                mouse_data = processed[processed['mouse']==mouse]

                day_range = list(range(1, max(mouse_data['exposure_day'])+1)) #To compare curr day with the day before
                mouse_p_values = []
                mouse_u_values = []

                for day in day_range:
                        spearmans_today = np.concatenate(mouse_data[mouse_data['exposure_day'] == day]['spearman_r_dist'].values)
                        spearmans_yesterday = np.concatenate(mouse_data[mouse_data['exposure_day'] == day - 1]['spearman_r_dist'].values)

                        res = mannwhitneyu(spearmans_today, spearmans_yesterday)

                        mouse_p_values.append(res.pvalue)
                        mouse_u_values.append(res.statistic)

                p_values.append(mouse_p_values)
                u_values.append(mouse_u_values)

        return p_values, u_values

In [ ]:
def plot_pvalues_over_days(p_values, mice):
    """
    Plot p-values over exposure days for each mouse.

    Parameters:
    - p_values: list of lists, one per mouse
    - mice: list of mouse IDs in same order as p_values
    """
    fig, ax = plt.subplots(figsize=(12, 4))

    for mouse_pvals, mouse in zip(p_values, mice):
        ax.plot(
            range(1, len(mouse_pvals) + 1),
            mouse_pvals,
            label=mouse,
            marker='o',
            linestyle='-',
        )

    ax.axhline(0.05, color='red', linestyle='--', linewidth=1, label='p = 0.05')
    ax.set_xlabel('Day')
    ax.set_ylabel('Mann-Whitney U p-value')
    ax.set_title('Day-to-Day Spearman r Distribution Changes (Mann-Whitney U)')
    ax.legend(title='Mouse')
    ax.set_yscale('log')  # Optional: log scale for better visibility if p-values vary widely
    ax.grid(True)
    plt.tight_layout()
    plt.show()

    return fig

In [ ]:
plot_pvalues_over_days(p_values, mice)

# Cells across days

Found through ROICaT, going to test what is going on with it

In [ ]:
def load_roicat_data(mouse):
    base_path = '/nemo/lab/znamenskiyp/home/shared/projects/colasa_3d-vision_revisions'
    roicat_path = os.path.join(base_path, mouse, 'ROICaT', f'{mouse}.tracking.results_clusters.json')
    
    with open(roicat_path, 'r') as f:
        roicat_dict = json.load(f)
    
    print(f"Loaded ROICaT data for {mouse}")
    print(f"Data type: {type(roicat_dict)}")
    if isinstance(roicat_dict, dict):
        print(f"Top-level keys: {list(roicat_dict.keys())}")
    
    return roicat_dict

Okay, now we load all sessions for a single mouse and add the cluster labels. 

We need to note down which session corresponds to which index in ROICaT

In [ ]:
roicat_sessions = {
    'PZAG16.3c': ['S20250219', 'S20250313']
}

In [ ]:
PROJECT='colasa_3d-vision_revisions'
flz_session = flz.get_flexilims_session(PROJECT)

processed_root = flz.get_data_root('processed', 
                                   project=PROJECT, 
                                   flexilims_session=flz_session
                                  )


In [ ]:
mouse = 'PZAG16.3c'

processed = learn.find_processed_sessions(mouse)

In [ ]:
processed

def select_roicat_sessions(processed, mouse, roicat_sessions):

    mouse_sessions = roicat_sessions[mouse]

    session_names = []
    for session in mouse_sessions:
        session_names.append(f'{mouse}_{session}')

    roicat_processed = processed[processed['name'].isin(session_names)]

    return roicat_processed

roicat_processed = select_roicat_sessions(processed, mouse, roicat_sessions)


def generate_roicat_neuronsdf(roicat_processed):
    roicat_neuronsdf = []
    for i, session in tqdm(roicat_processed.iterrows()):
        neurons_path = processed_root / session.path / 'neurons_df.pickle'

        with open(neurons_path, 'rb') as f:
            neurons_df = learn.NumpyFixUnpickler(f).load()
            roicat_neuronsdf.append(neurons_df)

    return roicat_neuronsdf

roicat_neuronsdf = generate_roicat_neuronsdf(roicat_processed)


In [ ]:
processed['name']

In [ ]:
roicat_dict = load_roicat_data(mouse)



In [ ]:
print(len(roicat_neuronsdf[0]))
len(roicat_dict['labels_bySession'][0])

# Macro comparisons